In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_ID = "/opt/gpudata/models/google/gemma-4-31B-it"

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
ds = load_dataset("/opt/gpudata/medical-o1-reasoning-SFT", "en", split="train")

In [3]:
def to_prompt(ex):
    return {
        "prompt": [
            {
                "role": "user",
                "content": ex["Question"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "reasoning": ex["Complex_CoT"],
                "content": ex["Response"],
            },
        ],
        "chat_template_kwargs": {"enable_thinking": True},
    }

prompt_ds = ds.map(to_prompt, remove_columns=ds.column_names)
print(tokenizer.apply_chat_template(prompt_ds[0]["prompt"] + prompt_ds[0]["completion"], enable_thinking=True, tokenize=False))

Map:   0%|          | 0/19704 [00:00<?, ? examples/s]

<bos><|turn>system
<|think|>
<turn|>
<|turn>user
Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?<turn|>
<|turn>model
<|channel>thought
Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?

But wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.

So, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?

Oh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.

Let's put this together: if a bl

In [4]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16)

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

In [6]:
model.model

Gemma4Model(
  (vision_tower): Gemma4VisionModel(
    (patch_embedder): Gemma4VisionPatchEmbedder(
      (input_proj): Linear(in_features=768, out_features=1152, bias=False)
    )
    (encoder): Gemma4VisionEncoder(
      (rotary_emb): Gemma4VisionRotaryEmbedding()
      (layers): ModuleList(
        (0-26): 27 x Gemma4VisionEncoderLayer(
          (self_attn): Gemma4VisionAttention(
            (q_proj): Gemma4ClippableLinear(
              (linear): Linear(in_features=1152, out_features=1152, bias=False)
            )
            (k_proj): Gemma4ClippableLinear(
              (linear): Linear(in_features=1152, out_features=1152, bias=False)
            )
            (v_proj): Gemma4ClippableLinear(
              (linear): Linear(in_features=1152, out_features=1152, bias=False)
            )
            (o_proj): Gemma4ClippableLinear(
              (linear): Linear(in_features=1152, out_features=1152, bias=False)
            )
            (q_norm): Gemma4RMSNorm()
            (k_norm

In [5]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="gemma4-31b-medical-o1",
    num_train_epochs=3,
    per_device_train_batch_size=1,  # test 2 at 4096 if memory allows
    gradient_accumulation_steps=8,  # 8 gpu × 1 × 8 = effective batch 64
    learning_rate=1e-5,  # full FT; ~5e-6–1e-5 is the sane band
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    max_grad_norm=1.0,
    bf16=True,
    tf32=True,
    max_length=4096,
    packing=False,  # off for long variable reasoning traces
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    # assistant_only_loss=True,  # mask user turn; train on thinking + answer
    completion_only_loss=True,
    dataset_kwargs={"add_special_tokens": False},  # template already adds <bos>
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="wandb",
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [6]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=prompt_ds,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/19704 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/19704 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/19704 [00:00<?, ? examples/s]

/home/songs1/miniforge3/envs/gemma/bin/x86_64-conda-linux-gnu-ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/songs1/miniforge3/envs/gemma/bin/x86_64-conda-linux-gnu-ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


In [7]:
x = trainer.train_dataset[0]
x.keys()

dict_keys(['prompt', 'completion', 'chat_template_kwargs', 'input_ids', 'labels'])

In [13]:
print(tokenizer.decode(x["input_ids"], skip_special_tokens=False))

<bos><|turn>system
<|think|>
<turn|>
<|turn>user
Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?<turn|>
<|turn>model
<|channel>thought
Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?

But wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.

So, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?

Oh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.

Let's put this together: if a bl

In [14]:
print(tokenizer.decode(x["labels"], skip_special_tokens=False))

OverflowError: out of range integral type conversion attempted

In [8]:
len(x["input_ids"])

567

In [10]:
len(x["labels"])

567

In [15]:
x["labels"]

[-100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 100,
 45518,
 107,
 19058,
 236764,
 1531,
 236789,
 236751,
 1460,
 1144,
 236789,
 236751,
 1771,
 580,
 1590,
 236761,
 1191,
 236789,
 560,
 2506,
 11059,
 22702,
 528,
 506,
 1589,
 236789,
 236751,
 2378,
 3774,
 532,
 2420,
 753,
 532,
 600,
 96564,
 2613,
 16669,
 236772,
 10619,
 236764,
 7463,
 496,
 15901,
 236881,
 108,
 4573,
 4491,
 236764,
 993,
 236789,
 236751,
 919,
 236761,
 669,
 1447,
 3718,
 2420,
 563,
 79471,
 532,
 21870,
 236764,
 837,
 563,
 1133,
 49117,
 496,
 2563,
 8134,
 573,
 5268,
 38281,
 110604,
 236764